In [1]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)
library_path

PosixPath('/mnt/DataVol/Beratungen/Yurttas/survival/src')

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats

In [3]:
# load data
DATA_PATH = "/mnt/DataVol/Beratungen/Yurttas/survival/data"
PLOTS_PATH = "/mnt/DataVol/Beratungen/Yurttas/survival/plots"

df = pd.read_csv(f"{DATA_PATH}/all_data.csv", sep="\t")

In [4]:
cols_to_use = ["Sex", "Age", "Tumor", "Zn HIPEC", "PreOP CTx", "Thermoablation", "sPCI", "pPCI"]
df = df[cols_to_use].copy()

In [5]:
# data wrangling
df['Sex'] = df['Sex']-1
df['PreOP CTx'] = df['PreOP CTx'].apply(lambda x: 1 if x >= 1 else x)

keep_tumor = [1, 5, 4, 6, 7, 3, 2]  # remove 8 if you decide to drop it

# Filter rows
df = df[df["Tumor"].isin(keep_tumor)].copy()
df['Tumor'] = df['Tumor'].apply(lambda x: f"type_{x}")

# --- 1. Rename columns ---
df = df.rename(columns={
    "Zn HIPEC"  : "Zn_HIPEC",
    "PreOP CTx" : "PreOP_CTx",
})

In [6]:
# --- 5. One-hot encode Tumor (most frequent as reference) ---
# Tumor "1" (n=122) is the natural reference category
df = pd.get_dummies(df, columns=["Tumor"], drop_first=False, dtype=int)
df = df.drop(columns=["Tumor_type_1"], inplace=False)  # explicitly set Tumor_1 as reference

# --- 6. Create outcome variables ---
df["raw_diff"] = df["sPCI"] - df["pPCI"]
df["abs_diff"] = np.abs(df["raw_diff"])

In [7]:
feature_cols = [
    'Sex', 'Age', 'Zn_HIPEC', 'PreOP_CTx', 'Thermoablation', 'Tumor_type_2', 'Tumor_type_3', 'Tumor_type_4', 'Tumor_type_5',
       'Tumor_type_6', 'Tumor_type_7'
]

X     = df[feature_cols]
y_raw = df["raw_diff"]
y_abs = df["abs_diff"]

print(f"Final feature matrix: {X.shape}")

Final feature matrix: (412, 11)


In [ ]:
# Add intercept (statsmodels requires explicit constant term, unlike sklearn)
X_with_const = sm.add_constant(X)  # Adds column of 1's for β₀

# Fit quantiles
quantiles = [0.1, 0.5, 0.9]
models = {}
predictions = {}

for q in quantiles:
    # QuantReg: y ~ X, optimize for quantile q
    model = sm.QuantReg(y_raw, X_with_const)
    result = model.fit(q=q)  # Uses linear programming under the hood
    models[q] = result
    predictions[q] = result.predict(X_with_const)

QR(τ=0.1): β₀=3.444, β₁=0.222
QR(τ=0.5): β₀=6.723, β₁=-0.482
QR(τ=0.9): β₀=18.206, β₁=-4.000


/tmp/ipykernel_17416/2728766287.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"QR(τ={q}): β₀={result.params[0]:.3f}, β₁={result.params[1]:.3f}")


In [ ]:
print(models[0.5].summary())

                         QuantReg Regression Results                          
Dep. Variable:               raw_diff   Pseudo R-squared:              0.07763
Model:                       QuantReg   Bandwidth:                       3.252
Method:                 Least Squares   Sparsity:                        16.61
Date:                Tue, 14 Apr 2026   No. Observations:                  412
Time:                        17:44:08   Df Residuals:                      400
                                        Df Model:                           11
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              3.4445      1.959      1.758      0.080      -0.408       7.297
Sex                0.2222      0.551      0.403      0.687      -0.861       1.306
Age               -0.0556      0.019     -2.989      0.003      -0.092      -0.019
Zn_HIPEC           0.3889      1